# Agent Factory

The `factory.py` module builds a compiled LangGraph agent that repeatedly calls a chat model, executes requested tools, applies middleware, and stops when the model no longer requests additional tool calls or structured-output processing.

The public API is centered on `create_agent`. The overloads preserve the type of the agent's structured response, while the concrete implementation builds the state graph, middleware nodes, model and tool nodes, routing edges, persistence configuration, cache integration, and stream transformers.

## Constants

1. `STRUCTURED_OUTPUT_ERROR_TEMPLATE`: Stores the default error message returned to the model when structured-output validation fails and the configured strategy allows a retry.

   * **Definition:**
     ```python
     STRUCTURED_OUTPUT_ERROR_TEMPLATE = (
         "Error: {error}\n"
         " Please fix your mistakes."
     )
     ```

2. `DYNAMIC_TOOL_ERROR_TEMPLATE`: Stores the diagnostic message raised when middleware adds client-side tools that were not registered when the agent was created.

   The message explains that the tool must either be registered through `create_agent`, exposed through `AgentMiddleware.tools`, or handled dynamically through `wrap_tool_call`.

   * **Type:**
     ```python
     DYNAMIC_TOOL_ERROR_TEMPLATE: str
     ```

3. `FALLBACK_MODELS_WITH_STRUCTURED_OUTPUT`: Stores regular-expression patterns for model names assumed to support provider-native structured output when model-profile information is unavailable.

   * **Type:**
     ```python
     FALLBACK_MODELS_WITH_STRUCTURED_OUTPUT: list[str]
     ```

### Functions

1. `create_agent`: Creates and compiles an agent graph that calls a chat model and executes tools in a loop.

   The agent adds the optional system message before model invocation. When the returned `AIMessage` contains tool calls, the graph routes to the tool node, appends the resulting `ToolMessage` values, and returns to the model. The loop finishes when no executable tool calls remain, a structured response is produced, a return-direct tool completes, or middleware redirects execution to the end.

   * **Syntax:**
     ```python
     create_agent(
         model: str | BaseChatModel, # Model identifier or initialized chat model
         tools: Sequence[
             BaseTool
             | Callable[
                 ...,
                 Any
             ]
             | dict[
                 str,
                 Any
             ]
         ] | None = None, # Client-side, callable, or provider-native tools
         *,
         system_prompt: str
         | SystemMessage
         | None = None, # Optional system instruction
         middleware: Sequence[
             AgentMiddleware[
                 StateT_co,
                 ContextT
             ]
         ] = (), # Middleware applied to the agent
         response_format: ResponseFormat[ResponseT]
         | type[ResponseT]
         | dict[
             str,
             Any
         ]
         | None = None, # Optional structured-output schema or strategy
         state_schema: type[
             AgentState[
                 ResponseT
             ]
         ] | None = None, # Optional custom agent-state schema
         context_schema: type[
             ContextT
         ] | None = None, # Optional runtime-context schema
         checkpointer: Checkpointer | None = None, # Per-thread state persistence
         store: BaseStore | None = None, # Cross-thread persistent storage
         interrupt_before: list[str] | None = None, # Nodes that interrupt before execution
         interrupt_after: list[str] | None = None, # Nodes that interrupt after execution
         debug: bool = False, # Enable detailed graph debugging
         name: str | None = None, # Name of the compiled graph
         cache: BaseCache[
             Any
         ] | None = None, # Optional graph-execution cache
         transformers: Sequence[
             TransformerFactory
         ] | None = None # Additional stream-transformer factories
     ) -> CompiledStateGraph[
         AgentState[
             ResponseT
         ],
         ContextT,
         InputAgentState,
         OutputAgentState[
             ResponseT
         ]
     ]
     ```

## Overloads

1. **Agent without structured output**

   When `response_format` is `None`, the structured-response type resolves to `Any`.

   * **Return type:**
     ```python
     CompiledStateGraph[
         AgentState[
             Any
         ],
         ContextT,
         InputAgentState,
         OutputAgentState[
             Any
         ]
     ]
     ```

2. **Agent with a raw dictionary schema**

   When `response_format` is a raw `dict[str, Any]`, the structured response is typed as `dict[str, Any]`.

   * **Return type:**
     ```python
     CompiledStateGraph[
         AgentState[
             dict[
                 str,
                 Any
             ]
         ],
         ContextT,
         InputAgentState,
         OutputAgentState[
             dict[
                 str,
                 Any
             ]
         ]
     ]
     ```

3. **Agent with a typed schema or strategy**

   When `response_format` is a response strategy or Python schema type, `ResponseT` is inferred from that schema.

   * **Return type:**
     ```python
     CompiledStateGraph[
         AgentState[
             ResponseT
         ],
         ContextT,
         InputAgentState,
         OutputAgentState[
             ResponseT
         ]
     ]
     ```

## Parameters

1. `model`: Accepts either a model identifier string or an initialized `BaseChatModel`.

   String identifiers are passed to `init_chat_model` before graph construction.

2. `tools`: Accepts `BaseTool` objects, Python callables, and provider-native tool dictionaries.

   Callables are converted by `ToolNode`. Dictionary tools are treated as provider-native tools and are bound to the model but are not executed by the client-side `ToolNode`.

   Middleware tools are combined with regular tools. When there are no client-side tools and no tool-call wrappers, the graph contains only the model and applicable middleware nodes.

3. `system_prompt`: Accepts a plain string or `SystemMessage`.

   Strings are converted into `SystemMessage` objects. The system message is prepended to the current message history before every model call.

4. `middleware`: Supplies agent middleware instances.

   Supported middleware stages include:

   - `before_agent` and `abefore_agent`
   - `before_model` and `abefore_model`
   - `wrap_model_call` and `awrap_model_call`
   - `after_model` and `aafter_model`
   - `after_agent` and `aafter_agent`
   - `wrap_tool_call` and `awrap_tool_call`

   Middleware may contribute tools, state schemas, stream transformers, model settings, tool choices, response formats, commands, and execution jumps.

5. `response_format`: Configures structured output.

   It may be:

   - `ToolStrategy`
   - `ProviderStrategy`
   - `AutoStrategy`
   - A Python schema type
   - A raw schema dictionary

   Raw schemas are wrapped in `AutoStrategy`. During model execution, automatic strategy selection chooses provider-native structured output when supported and otherwise falls back to tool-based structured output.

6. `state_schema`: Supplies a custom state schema.

   Middleware state schemas are merged in middleware-registration order. The explicitly supplied state schema is processed last, so its field annotations override conflicting middleware fields.

   Input and output schemas are derived from the merged state schema while respecting `OmitFromSchema` metadata.

7. `context_schema`: Defines the runtime context type available through `Runtime[ContextT]`.

8. `checkpointer`: Adds thread-level state persistence, such as conversation memory.

9. `store`: Adds persistent storage shared across threads or conversations.

10. `interrupt_before`: Configures graph nodes that pause before execution.

11. `interrupt_after`: Configures graph nodes that pause after execution.

12. `debug`: Enables detailed graph execution and transition output.

13. `name`: Names the compiled graph.

    The name is also added to LangSmith metadata under `lc_agent_name` and assigned to generated model output messages.

14. `cache`: Configures graph-level execution caching.

15. `transformers`: Adds stream-transformer factories after the built-in and middleware-provided transformers.

## Model Binding

For each model invocation, the factory creates a `ModelRequest` containing:

- The current model.
- Current tools.
- The system message.
- The active response format.
- Message history.
- Tool-choice settings.
- Complete agent state.
- Runtime context.

Middleware may replace or modify these values before the request reaches the model.

The final model binding depends on the effective response strategy:

1. With `ProviderStrategy`, provider-specific structured-output arguments are applied through `bind_tools`.

2. With `ToolStrategy`, structured-output tools are appended and tool choice is forced to `"any"` when structured-output tools exist.

3. Without structured output, tools are bound normally when present.

4. Without any tools, model settings are applied through `bind`.

For OpenAI-compatible models using provider-native structured output outside the Responses API, `strict=True` is added for backward compatibility.

## Structured Output

### Provider Strategy

When provider-native structured output is active, the model message is parsed with `ProviderStrategyBinding`.

A parsing failure is wrapped in `StructuredOutputValidationError`. A successful parse is stored in the agent state's `structured_response` field.

### Tool Strategy

Tool-based structured output is represented through synthetic tools generated from the response schema.

When exactly one structured-output tool is called:

- Its arguments are parsed.
- A confirming `ToolMessage` is appended.
- The parsed value is stored in `structured_response`.

When multiple structured-output tools are called, `MultipleStructuredOutputsError` is created. Depending on the strategy's error-handling configuration, the error is either raised or converted into corrective tool messages so the model can retry.

Validation failures follow the same retry-or-raise behaviour through `StructuredOutputValidationError`.

Middleware may narrow a `ToolStrategy` to tools declared by the original response format, but it cannot introduce new structured-output tools after agent construction.

## Middleware Composition

Synchronous and asynchronous wrappers are composed separately.

The first registered wrapper is the outermost layer. Each wrapper receives a handler representing the remaining inner layers and may call it once, multiple times, or not at all.

Commands returned by model-call middleware are preserved in inner-to-outer order. The first graph command updates messages and, when present, `structured_response`. Additional middleware commands are appended without merging.

Middleware commands using `goto`, `resume`, or `graph` through `wrap_model_call` are not supported and raise `NotImplementedError`. Middleware should use the `jump_to` state field for supported routing changes.

Middleware wrapper traces exclude runtime and handler objects before being sent to LangSmith.

## Tool Execution

Client-side tools are registered in `ToolNode`. Provider-native dictionary tools are bound to the model but are not included in client-side execution.

Dynamic client-side tools inserted by middleware must either:

- Already exist in the tool node, or
- Be handled through `wrap_tool_call` or `awrap_tool_call`.

Otherwise, `create_agent` raises a `ValueError` containing the dynamic-tool diagnostic message.

Tool execution may route:

- Back to the model loop.
- To the final agent exit for return-direct tools.
- To structured-output completion.
- Through middleware-controlled jumps.

## Graph Structure

The compiled graph may contain these nodes:

- `"model"`
- `"tools"`
- `<middleware-name>.before_agent`
- `<middleware-name>.before_model`
- `<middleware-name>.after_model`
- `<middleware-name>.after_agent`

Execution order follows these rules:

1. `before_agent` middleware runs once before the agent loop.

2. `before_model` middleware runs at the start of each loop iteration.

3. The model node invokes the bound chat model.

4. `after_model` middleware runs after each model invocation.

5. Tool calls route to the tool node and then back to the loop entry.

6. `after_agent` middleware runs once when the agent is finishing.

7. The graph transitions to `END`.

Before-stage middleware is chained in registration order. After-stage middleware is traversed in reverse registration order.

## Compilation Configuration

The graph is compiled with:

- The supplied checkpointer.
- The supplied store.
- Before-node and after-node interrupts.
- Debug configuration.
- Graph name.
- Graph cache.
- Built-in and custom stream transformers.

The transformer order is:

1. `ToolCallTransformer`
2. `SubagentTransformer`
3. Transformers supplied by middleware
4. Transformers supplied through `transformers`

The returned graph receives this default Runnable configuration:

```python
{
    "recursion_limit": 9_999,
    "metadata": {
        "ls_integration": "langchain_create_agent",
    },
}
```

When `name` is provided, `metadata["lc_agent_name"]` is also added.

## Errors

`create_agent` may raise:

1. `AssertionError`: Duplicate middleware names were supplied.

2. `ValueError`: Middleware introduced an unknown client-side tool without a tool-call wrapper.

3. `ValueError`: A dynamically selected `ToolStrategy` refers to a structured-output tool that was not declared in the original response format.

4. `StructuredOutputValidationError`: Provider-native or tool-based structured output failed validation and retry handling was disabled.

5. `MultipleStructuredOutputsError`: More than one structured-output tool was called and retry handling was disabled.

6. `NotImplementedError`: Model-call middleware returned unsupported command fields such as `goto`, `resume`, or `graph`.

7. Provider, tool, middleware, graph-compilation, persistence, and cache exceptions are allowed to propagate.

## Private Components Omitted

The following implementation details are private and are not documented as standalone API entries:

- Middleware wrapper-composition helpers
- Schema-resolution helpers
- Model-capability detection helpers
- Structured-output error helper
- Conditional routing functions
- Middleware edge-construction helper
- Trace-input scrubbing helper
- Internal composed model-response dataclass